# Cubo G7B — laboratorio de consultas
Descarga una versión verificada de Azure y abre DuckDB en solo lectura.
En las variables de entorno de Deepnote configura **AZURE_LAGO_URL** con acceso de solo lectura al contenedor. No pegues la URL firmada en una celda.
Los resultados en revisión se muestran para explorar; no constituyen recomendaciones de Geoda.

In [ ]:
%pip install duckdb==2.0.0 "azure-storage-blob>=12.20,<13"

In [ ]:
import os, sys, tempfile, subprocess
from pathlib import Path
if not os.environ.get("AZURE_LAGO_URL"):
    raise RuntimeError("Configura AZURE_LAGO_URL como variable de entorno secreta y reinicia el entorno.")
carpeta = Path(tempfile.mkdtemp(prefix="g7b_cubo_"))
repo = carpeta / "repositorio"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
               "https://github.com/Maximiliano-cabello-mata/amatista-g7b-etl.git", str(repo)], check=True, capture_output=True)
sys.path.insert(0, str(repo))
revision = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
print("Código de consulta:", revision)

In [ ]:
from g7b_lago import Lago
from g7b_almacen.descargar import descargar
import duckdb
ruta = carpeta / "cubo.duckdb"
verificacion = descargar(Lago.desde_entorno(), ruta)
display(verificacion)
conexion = duckdb.connect(str(ruta), read_only=True)
def consultar(sql):
    resultado = conexion.execute(sql)
    columnas = [c[0] for c in resultado.description]
    return [dict(zip(columnas, fila)) for fila in resultado.fetchall()]

In [ ]:
display(consultar("SELECT * FROM v_calidad ORDER BY fuente, estado"))

In [ ]:
display(consultar("""SELECT fuente, version, escena, prueba, metrica, unidad,
estado, mediciones, mediana FROM v_resumen ORDER BY mediciones DESC LIMIT 20"""))

## Solo mediciones aceptadas
Aceptado significa que pasó los controles técnicos de captura. No demuestra que el equipo represente a usuarios ni que se pueda recomendar un perfil.

In [ ]:
display(consultar("SELECT version, prueba, unidad, valor FROM v_mediciones_aceptadas LIMIT 20"))

In [ ]:
conexion.close()